query1: SELECT A.name FROM A LEFT JOIN B ON A.id = B.id WHERE B.price >= 3;

query2: SELECT A.name FROM A LEFT JOIN B ON A.id = B.id AND B.price >= 3;


In this notebook I'll show how the new implementation would encode these two quries 
(same as query3 and 4 from join.ipynb)

In [20]:
from z3 import *

# todo, explain what present means 
A_present = Bool("A_present")
A_id = Int("A_id")
A_name = String("A_name")

B_present = Bool("B_present")
B_id = Int("B_id")
B_price = Int("B_price") 

# define NULL 
NULL = IntVal(-1) 

Encode q1:
SELECT A.name, B.price
FROM A LEFT JOIN B ON A.id = B.id
WHERE B.price >= 3

In [21]:
q1_constraints = []

# step1: encode join 
q1_match = Bool("q1_match")         # matched row (A,B)
q1_right_null = Bool("q1_right_null") # null-extended (A,NULL)

q1_join_cond = And(A_present, B_present, A_id == B_id)

q1_constraints += [
    q1_match == q1_join_cond,
    q1_right_null == And(A_present, Not(q1_join_cond))
]


# step2: encode where 
# WHERE B.price >= 3 eliminates null rows because B_price is undefined in null rows
q1_after_match = Bool("q1_after_match")
q1_after_right_null = Bool("q1_after_right_null")

q1_constraints += [
    q1_after_match == And(q1_match, B_price >= 3),
    q1_after_right_null == False   # null rows eliminated by WHERE B.price >= 3
]


# # step3: projection: SELECT A.name, B.price
# # but since we only focus on some key operations, we can savely skip this step.
# q1_proj_match_name  = String("q1_proj_match_name")
# q1_proj_match_price = Int("q1_proj_match_price")
# q1_proj_null_name   = String("q1_proj_null_name")
# q1_proj_null_price  = Int("q1_proj_null_price")

# q1_constraints += [
#     q1_proj_match_name  == If(q1_after_match, A_name, ""),
#     q1_proj_match_price == If(q1_after_match, B_price, 0),

#     q1_proj_null_name   == "",   # null row eliminated by WHERE
#     q1_proj_null_price  == 0
# ]

Encode q2:
SELECT A.name
FROM A LEFT JOIN B ON A.id = B.id AND B.price >= 3


In [23]:
# a more general version 
q2_constraints = []

# step1: encode join
q2_match = Bool("q2_match")
q2_null  = Bool("q2_null")

q2_join_cond = And(And(A_present, B_present, A_id == B_id), B_price >= 3)

q2_constraints += [
    q2_match == q2_join_cond,
    q2_null == And(A_present, Not(q2_match))
]

# step2: encode where (no where clause)
q2_after_match = Bool("q2_after_match")
q2_after_right_null = Bool("q2_after_right_null")
q2_where_cond = BoolVal(True)
q2_constraints += [
    q2_after_match == And(q1_match, q2_where_cond),
    q2_after_right_null == And(q2_null, q2_where_cond)  
]


# # same as above, skip
# # Projection
# q2_proj_match = String("q2_proj_match")
# q2_proj_null  = String("q2_proj_null")

# q2_constraints += [
#     q2_proj_match == If(q2_match, A_name, ""),
#     q2_proj_null  == If(q2_null, A_name, "")
# ]

Non-equivalence: outputs differ

In [24]:
s = Solver()
s.add(q1_constraints + q2_constraints)

diff = Or(
    q1_after_match != q2_match,
    q1_after_right_null != q2_null,
    # q1_proj_match != q2_proj_match,
    # q1_proj_null != q2_proj_null
)

s.add(diff)

print("SAT? ->", s.check())
if s.check() == sat:
    m = s.model()
    for v in m:
        print(v, "=", m[v])
else:
    print("Queries are equivalent.")

SAT? -> sat
B_present = False
q2_null = True
A_id = 0
q2_match = False
A_present = True
B_price = 2
q1_match = False
q1_after_match = False
q1_right_null = True
B_id = 0
q2_after_match = False
q1_after_right_null = False
q2_after_right_null = True


General encoding technique for different kind of joins

In [ ]:
match = BoolVal(True) 

# No Join (only one table A): 
# match 


# Inner Join:
inner_match      = match
inner_right_null = False
inner_left_null  = False


# A Left Join B :
left_match      = match
left_right_null = And(A_present, Not(match))   # A kept, B missing
left_left_null  = False                     # LEFT JOIN never produces NULL-A


# A Right Join B :
right_match      = match
right_right_null = False                    # RIGHT JOIN never produces NULL-B
right_left_null  = And(B_present, Not(match))  # B kept, A missing


# Full Join: 
full_match      = match
full_right_null = And(A_present, Not(match)) # A row unmatched
full_left_null  = And(B_present, Not(match)) # B row unmatched


# Cartisian Product: 
cp_match = And(A_present, B_present) 
cp_left_null = False
cp_right_null = False